# 01 · Bronze — raw ingestion into Delta

**Goal:** land the five raw CSV extracts *exactly as received* into Delta tables, adding lineage columns
(`_source_file`, `_ingest_ts`, `_batch_id`). No typing, no cleaning — that is the Silver layer's job.
Keeping Bronze raw means every downstream fix is reproducible and auditable.

```
raw CSV  ──►  bronze_sellers
              bronze_accounts
              bronze_opportunities
              bronze_opportunity_stage_history
              bronze_activities
```

Runs on Databricks, Microsoft Fabric or local PySpark (see the configuration cell).

## 0. Configuration

The same notebook runs unchanged on **Databricks** (Free Edition or any workspace), **Microsoft Fabric**
(attached to a Lakehouse) and a **local PySpark + Delta Lake** session. The platform is auto-detected,
or you can force it with the `LAKEHOUSE_PLATFORM` environment variable.

| Platform | Raw files | Tables |
|---|---|---|
| Databricks | Unity Catalog volume `/Volumes/workspace/sales_lakehouse/raw` | `workspace.sales_lakehouse.<table>` |
| Fabric | Lakehouse `Files/raw` | Lakehouse `Tables` (Delta) |
| Local | `../data/raw` | Spark database `sales_lakehouse` |

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
import os
from datetime import datetime, timezone

def _detect_platform() -> str:
    if os.environ.get("LAKEHOUSE_PLATFORM"):
        return os.environ["LAKEHOUSE_PLATFORM"].lower()
    if "DATABRICKS_RUNTIME_VERSION" in os.environ:
        return "databricks"
    if os.path.isdir("/lakehouse/default"):          # Fabric notebook with a default Lakehouse attached
        return "fabric"
    return "local"

PLATFORM = _detect_platform()            # "databricks" | "fabric" | "local"
CATALOG  = "workspace"                   # Databricks Unity Catalog (Free Edition default catalog)
SCHEMA   = "sales_lakehouse"             # Databricks schema / local Spark database

if PLATFORM == "databricks":
    RAW_PATH    = f"/Volumes/{CATALOG}/{SCHEMA}/raw"
    EXPORT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/export"
    def tbl(name: str) -> str:
        return f"{CATALOG}.{SCHEMA}.{name}"
elif PLATFORM == "fabric":
    RAW_PATH    = "Files/raw"
    EXPORT_PATH = "Files/export"
    def tbl(name: str) -> str:
        return name                       # tables live in the attached Lakehouse
else:
    RAW_PATH    = os.path.abspath("../data/raw")
    EXPORT_PATH = os.path.abspath("../lakehouse/export")
    def tbl(name: str) -> str:
        return f"{SCHEMA}.{name}"

try:
    spark  # noqa: F821 - pre-defined on Databricks and Fabric
except NameError:                          # local run: build a Delta-enabled SparkSession
    from pyspark.sql import SparkSession
    from delta import configure_spark_with_delta_pip
    _builder = (SparkSession.builder.appName("sales-pipeline-lakehouse")
                .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
                .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
                .config("spark.sql.warehouse.dir", os.path.abspath("../lakehouse/warehouse")))
    spark = configure_spark_with_delta_pip(_builder).getOrCreate()

if PLATFORM == "databricks":
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
    for _vol in ("raw", "export"):
        try:
            spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{_vol}")
        except Exception as _e:              # no privilege → create the volume in Catalog Explorer instead
            print(f"could not create volume '{_vol}': {str(_e).splitlines()[0]}")
elif PLATFORM == "local":
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {SCHEMA}")

RUN_TS = datetime.now(timezone.utc)
print(f"platform={PLATFORM} | raw={RAW_PATH} | export={EXPORT_PATH} | spark={spark.version}")

## 1. Ingest each source file

* `inferSchema=false` → every column stays a string in Bronze (raw fidelity).
* `_metadata.file_path` records which file each row came from.
* `mode("overwrite")` because each run re-lands the full extract; an incremental feed would use `append`
  partitioned by `_batch_id`.

In [ ]:
from pyspark.sql import functions as F

SOURCES = ["sellers", "accounts", "opportunities", "opportunity_stage_history", "activities"]
BATCH_ID = RUN_TS.strftime("%Y%m%d%H%M%S")

def ingest_bronze(source: str) -> int:
    """Land one raw CSV as-is (all columns as strings) into a Delta table with lineage columns."""
    df = (spark.read
            .option("header", "true")
            .option("inferSchema", "false")          # keep raw strings; typing happens in Silver
            .option("quote", '"')
            .option("escape", '"')
            .csv(f"{RAW_PATH}/{source}.csv")
            .withColumn("_source_file", F.col("_metadata.file_path"))
            .withColumn("_ingest_ts", F.lit(RUN_TS))
            .withColumn("_batch_id", F.lit(BATCH_ID)))
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(tbl(f"bronze_{source}")))
    return spark.table(tbl(f"bronze_{source}")).count()

for src in SOURCES:
    rows = ingest_bronze(src)
    print(f"bronze_{src:<28} {rows:>8,} rows")

## 2. Inspect what landed

Notice the raw problems that Silver will have to deal with: inconsistent stage labels, lower-case
currency codes, missing amounts and mixed date formats.

In [ ]:
bronze_opps = spark.table(tbl("bronze_opportunities"))
bronze_opps.printSchema()

print("Distinct raw stage labels:")
bronze_opps.groupBy("stage").count().orderBy(F.desc("count")).show(20, truncate=False)

print("Rows with a non-ISO expected_close_date:")
bronze_opps.filter(F.col("expected_close_date").contains("/")).select("opportunity_id", "expected_close_date").show(5)

## 3. Delta transaction log

Every write is a versioned commit — the basis for time travel, auditing and rollback.

In [ ]:
(spark.sql(f"DESCRIBE HISTORY {tbl('bronze_opportunities')}")
      .select("version", "timestamp", "operation", "operationMetrics")
      .show(5, truncate=False))

✅ **Bronze complete.** Continue with `02_silver_clean_conform.ipynb`.